In [1]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
sys.path.append('/Users/kevinlaventure/python_code')
from python_module.pricing_model import SABRModel
from scipy.optimize import minimize, LinearConstraint

# Configure pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:_.2f}')

In [2]:
def compute_option_surface(
    F: float, 
    K_list: list, 
    T_list: list, 
    alpha_list: list, 
    beta: float, 
    rho: float, 
    nu: float,
    r: float, 
    slide_scenario=None,
    slide_type: str = 'spot_only', 
    slide_compute: str = 'option_pnl',
    compute_bs_greeks: bool = True, 
    compute_model_greek: bool = False
) -> pd.DataFrame:
    """
    Computes SABR option prices and Greeks over a grid of strikes and maturities.
    
    Args:
        F: Forward price
        K_list: List of strike prices
        T_list: List of times to maturity (in years)
        alpha_list: List of alpha (volatility) parameters
        beta, rho, nu: SABR parameters
        r: Risk-free rate
        slide_scenario: List of spot bumps (optional)
        slide_type: 'spot_vol' or 'spot_only'
        slide_compute: PnL calculation type ('delta_hedged_pnl', 'option_pnl', 'delta_pnl')
        compute_bs_greeks: If True, returns Black-Scholes Greeks
        compute_model_greek: If True, returns SABR model Greeks
        
    Returns:
        DataFrame with rows for each (K, T, alpha) combination containing:
        - Input parameters: F, K, T, alpha, beta, rho, nu, r, option_type
        - IV: Implied volatility
        - price: Option price
        - Greeks: delta, gamma, vega, theta, vanna, volga
        - Model Greeks (if compute_model_greek=True): sabr_delta, sabr_gamma, sabr_vega, sabr_vanna, sabr_volga, sabr_theta
        - Slides: PnL or price differences for each slide scenario
        
    Note:
        Option type is determined automatically: call if K > F, put if K ≤ F
    """
    results = []
    
    for i in range(len(alpha_list)):
        alpha = alpha_list[i]
        T = T_list[i]
        for K in K_list:
            # Determine option type: call if K > F, put otherwise
            option_type = 'call' if K > F else 'put'
            
            result = SABRModel.compute_option(
                F=F, 
                K=K, 
                T=T, 
                alpha=alpha, 
                beta=beta, 
                rho=rho, 
                nu=nu,
                r=r, 
                option_type=option_type, 
                slide_scenario=slide_scenario,
                slide_type=slide_type, 
                slide_compute=slide_compute,
                compute_bs_greeks=compute_bs_greeks, 
                compute_model_greek=compute_model_greek)
                
            # Build row with inputs and results
            row = {
                'F': F,
                'K': K,
                'T': T,
                'alpha': alpha,
                'beta': beta,
                'rho': rho,
                'nu': nu,
                'r': r,
                'option_type': option_type,
            }
            
            # Add all result fields
            row.update(result)
            
            results.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Reorder columns: inputs first, then IV and price, then greeks, then slides
    input_cols = ['F', 'K', 'T', 'alpha', 'beta', 'rho', 'nu', 'r', 'option_type']
    price_cols = ['IV', 'price']
    greek_cols = ['delta', 'gamma', 'vega', 'theta', 'vanna', 'volga']
    sabr_greek_cols = ['sabr_delta', 'sabr_gamma', 'sabr_vega', 'sabr_vanna', 'sabr_volga', 'sabr_theta']
    
    # Build column order
    col_order = input_cols + price_cols
    col_order += [c for c in greek_cols if c in df.columns]
    col_order += [c for c in sabr_greek_cols if c in df.columns]
    
    # Add slide columns (remaining columns)
    slide_cols = [c for c in df.columns if c not in col_order]
    col_order += slide_cols
    
    # Reorder dataframe
    df = df[col_order]
    
    return df

In [12]:
# Generate surface with slides
F = 100.0  # Forward price
K_max = 100
K_min = 70
K_step = 1
K_list = np.arange(K_min, K_max + K_step, K_step) # 9 strikes from 80 to 120 --- IGNORE ---
T_list = [10/250, 20/250]  # Times to maturity
alpha_list = [0.1, 0.15]  # Alpha (volatility) parameters

beta = 1
rho = -0
nu = 0.0001
r = 0.0

# Compute option surface with slides
slide_scenario = [-0.3, -0.1, -0.05, -0.04, -0.03, -0.02, -0.01, 0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1]
df_surface = compute_option_surface(
    F=F,
    K_list=K_list,
    T_list=T_list,
    alpha_list=alpha_list,
    beta=beta,
    rho=rho,
    nu=nu,
    r=r,
    slide_scenario=slide_scenario,
    compute_bs_greeks=True,
    compute_model_greek=False
)
df_surface['symbol'] = df_surface['K'].astype(str) + '_' + df_surface['T'].astype(str)
df_surface.set_index('symbol', inplace=True)
df =  df_surface.loc[:, slide_scenario]

In [13]:
df_surface.head()

,F,K,T,alpha,beta,rho,nu,r,option_type,IV,price,delta,gamma,vega,theta,vanna,volga,-0.30,-0.10,-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.10
symbol,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
70_0.04,100.00,70,0.04,0.10,1,0,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.53,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
71_0.04,100.00,71,0.04,0.10,1,0,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,1.18,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
72_0.04,100.00,72,0.04,0.10,1,0,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,2.04,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
73_0.04,100.00,73,0.04,0.10,1,0,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,3.01,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
74_0.04,100.00,74,0.04,0.10,1,0,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,4.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00


In [14]:
df.tail()

,-0.30,-0.10,-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.10
symbol,,,,,,,,,,,,,,
96_0.08,25.63,5.73,1.75,1.21,0.77,0.42,0.16,-0.03,-0.16,-0.24,-0.30,-0.33,-0.35,-0.37
97_0.08,26.42,6.47,2.20,1.56,1.02,0.58,0.23,-0.03,-0.22,-0.36,-0.44,-0.50,-0.53,-0.58
98_0.08,27.14,7.17,2.66,1.93,1.30,0.76,0.31,-0.04,-0.30,-0.50,-0.63,-0.72,-0.78,-0.86
99_0.08,27.77,7.78,3.10,2.30,1.58,0.94,0.40,-0.04,-0.39,-0.66,-0.85,-0.99,-1.08,-1.22
100_0.08,28.31,8.31,3.52,2.65,1.85,1.13,0.50,-0.04,-0.49,-0.84,-1.11,-1.31,-1.44,-1.68


In [15]:
def compute_hedges(selection_short_scaled, df, short_leg_symbol, q_short):
    res_dict = dict()
    for long_leg_symbol in df.index:
        if long_leg_symbol == short_leg_symbol:
            continue
        selection_long = df.loc[long_leg_symbol]

        pos = selection_long > 0
        neg = selection_long < 0
        zero = selection_long == 0

        # Feasibility check on zero-slope indices
        if not (selection_short_scaled[zero] > 0).all():
            raise ValueError("Infeasible: selection <= 0 where selection_long == 0")

        L = (-selection_short_scaled[pos] / selection_long[pos]).max() if pos.any() else -np.inf
        U = (-selection_short_scaled[neg] / selection_long[neg]).min() if neg.any() else  np.inf

        if L >= U:
            continue
            raise ValueError("Infeasible: lower bound exceeds upper bound")

        q_long = L + 1e-9 # or any small epsilon
        selection_long_scaled = selection_long * q_long
        selection_combined_scaled = selection_short_scaled + selection_long_scaled 
        res_dict[(short_leg_symbol, long_leg_symbol)] = {'q_short': q_short, 'q_long': q_long, **selection_combined_scaled.to_dict()}
    return res_dict

In [16]:
res = dict()
for short_leg_symbol in df.index:

    selection_short = df.loc[short_leg_symbol]
    q_short = -30_000_000 / selection_short[-0.3]
    selection_short_scaled = selection_short * q_short

    res_temp = compute_hedges(selection_short_scaled, df, short_leg_symbol, q_short)
    res.update(res_temp)

In [18]:
res_df = pd.DataFrame(res).T
res_df = res_df.reset_index(names=['short_symbol', 'long_symbol'])
df_surface['ask_price'] = df_surface['vega'] + df_surface['price']
df_surface['bid_price'] = -df_surface['vega'] + df_surface['price']
res_df['bid_price'] = res_df['short_symbol'].map(df_surface['bid_price'])
res_df['ask_price'] = res_df['long_symbol'].map(df_surface['ask_price'])
res_df['e_pnl'] = (res_df['q_short'] * res_df['bid_price'] * -1) - (res_df['q_long'] * res_df['ask_price'])
res_df = res_df[res_df['e_pnl']>0].sort_values('e_pnl', ascending=False).reset_index(drop=True)
res_df['ratio'] = -res_df['q_short'] / res_df['q_long']
res_df = res_df.loc[res_df['ratio']<2]

In [19]:
res_df

,short_symbol,long_symbol,q_short,q_long,-0.30,-0.10,-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.10,bid_price,ask_price,e_pnl,ratio
0,100_0.08,100_0.04,-1_059_787.63,1_027_322.37,0.00,642_116.70,593_050.86,488_094.51,341_122.93,177_339.88,46_868.84,3_325.37,69_123.41,220_088.31,404_889.06,577_524.21,714_504.77,957_341.24,1.58,0.88,772_436.22,1.03
1,99_0.08,99_0.04,-1_080_341.45,1_215_690.63,4_778_408.12,2_056_179.52,1_043_540.22,731_791.79,412_005.20,146_048.79,0.00,1_388.90,120_115.63,292_580.45,461_728.20,597_825.61,695_031.53,845_221.16,1.12,0.46,650_312.19,0.89
2,98_0.08,99_0.04,-1_105_458.26,1_048_659.81,0.00,1_104_882.40,853_245.16,640_538.11,394_109.52,170_044.53,30_171.97,4_312.40,72_750.14,186_084.85,299_142.77,388_686.39,450_608.53,537_996.22,0.76,0.46,358_100.21,1.05
3,97_0.08,98_0.04,-1_135_461.72,1_391_719.06,8_742_964.76,3_556_890.31,1_504_215.20,964_348.91,484_913.02,150_373.63,0.00,3_801.73,90_562.11,195_057.69,282_289.59,343_414.96,382_148.62,430_693.87,0.49,0.21,268_805.08,0.82
4,96_0.08,97_0.04,-1_170_592.48,2_398_409.30,34_625_688.57,9_954_639.22,2_913_607.82,1_656_621.34,729_083.35,197_107.85,0.00,2_036.36,78_281.66,158_008.97,217_329.91,255_407.18,277_949.57,303_387.72,0.30,0.08,163_498.77,0.49
5,96_0.08,98_0.04,-1_170_592.48,1_077_655.57,0.00,1_743_988.19,1_047_366.01,702_097.76,372_244.66,129_869.25,12_845.57,5_010.29,56_978.20,122_285.12,176_174.85,212_774.08,235_011.80,260_394.41,0.30,0.21,128_501.43,1.09
6,95_0.08,96_0.04,-1_211_030.72,5_037_684.81,100_904_781.92,24_164_847.68,5_005_210.63,2_499_483.48,953_101.26,222_671.14,0.00,1_332.16,61_373.72,116_456.84,153_545.25,175_628.54,187_928.92,200_494.21,0.17,0.02,87_578.18,0.24
7,95_0.08,97_0.04,-1_211_030.72,1_653_944.21,14_565_906.03,5_500_040.11,1_802_642.16,1_033_027.30,454_614.71,121_402.57,0.00,4_201.85,53_526.32,103_019.95,138_411.43,160_130.76,172_372.53,184_929.75,0.17,0.08,80_308.59,0.73
8,94_0.08,95_0.04,-1_256_935.48,13_119_017.56,297_933_568.16,60_345_931.74,8_167_266.32,3_497_481.76,1_132_421.67,225_268.63,0.00,1_075.79,43_995.22,78_880.56,100_386.82,112_359.36,118_653.69,124_490.78,0.09,0.01,39_062.94,0.10
9,94_0.08,96_0.04,-1_256_935.48,3_290_519.28,55_504_497.56,14_472_445.61,3_100_392.05,1_546_506.38,586_059.14,134_849.95,0.00,2_786.18,39_845.42,72_393.66,93_355.68,105_238.61,111_521.86,117_357.81,0.09,0.02,37_945.56,0.38
